# Construcció del dataset integrat - BBF-14

Aquest notebook combina les mètriques procedents de les tres fonts de càlcul en un únic CSV que constituirà el dataset de treball per a l'anàlisi estadística.

Les tres fonts que s'integren són els millors resultats d'AlphaFold3 per complex (af3_scores_best.csv), la mitjana de les mètriques de Rosetta Relax calculades a partir dels fitxers .sc de cada complex (111 estructures per complex) i la mitjana de les mètriques d'interfície calculades per analyseRosettaCalculation (fitxers .csv de la carpeta binding_energy). Per a Rosetta i per a les mètriques d'interfície s'utilitza la mitjana aritmètica de totes les estructures generades per cada complex, ja que la variabilitat entre les poses optimitzades és reduïda i la mitjana és un estimador robust del valor central de cada mètrica.

La integració es fa mitjançant un join intern sobre la columna model, de manera que només s'inclouen els complexos per als quals existeixen resultats de les tres fonts. El dataset final conté 14 complexos i 50 columnes de mètriques.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import pandas as pd


In [2]:
BASE = Path('/Users/bertaguiu/projects/Scripts')
AF3_BEST_CSV = BASE / 'af3_tables' / 'af3_scores_best.csv'
ROSETTA_OUTPUT_MODELS = BASE / 'rosetta_relax' / 'output_models'
BINDING_ENERGY_DIR = BASE / 'rosetta_relax' / '.analysis' / 'binding_energy'
OUTPUT_CSV = BASE / 'final_bbf14.csv'


In [3]:
def _mean_numeric_columns(df: pd.DataFrame, drop_columns: Iterable[str] = ()) -> pd.Series:
    working = df.drop(columns=list(drop_columns), errors='ignore').copy()
    numeric = working.apply(pd.to_numeric, errors='coerce')
    return numeric.mean(axis=0, numeric_only=True)


def _read_rosetta_score_file(score_file: Path) -> pd.DataFrame:
    with score_file.open() as handle:
        lines = [line.strip() for line in handle if line.strip()]

    if len(lines) < 3:
        raise ValueError(f'Rosetta score file is too short: {score_file}')

    header_line = None
    data_lines = []
    for line in lines:
        if not line.startswith('SCORE:'):
            continue
        payload = line[len('SCORE:'):].strip()
        if payload.startswith('total_score'):
            header_line = payload
            continue
        data_lines.append(payload)

    if header_line is None:
        raise ValueError(f'Could not find SCORE header in {score_file}')
    if not data_lines:
        raise ValueError(f'Could not find SCORE rows in {score_file}')

    columns = header_line.split()
    rows = [row.split() for row in data_lines]
    return pd.DataFrame(rows, columns=columns)


def _summarize_rosetta_scores() -> pd.DataFrame:
    rows = []
    for model_dir in sorted(ROSETTA_OUTPUT_MODELS.iterdir()):
        if not model_dir.is_dir():
            continue

        score_file = model_dir / f'{model_dir.name}_relax.sc'
        if not score_file.exists():
            raise FileNotFoundError(f'Missing Rosetta score file: {score_file}')

        score_df = _read_rosetta_score_file(score_file)
        means = _mean_numeric_columns(score_df, drop_columns=('description',))
        row = {'model': model_dir.name}
        row.update({f'rosetta_{column}_mean': value for column, value in means.items()})
        row['rosetta_n_structures'] = len(score_df)
        rows.append(row)

    return pd.DataFrame(rows).sort_values('model').reset_index(drop=True)


def _summarize_binding_energy() -> pd.DataFrame:
    rows = []
    for csv_file in sorted(BINDING_ENERGY_DIR.glob('Complex*.csv')):
        binding_df = pd.read_csv(csv_file)
        if binding_df.empty:
            raise ValueError(f'Binding-energy CSV is empty: {csv_file}')

        model = binding_df['Model'].iloc[0]
        means = _mean_numeric_columns(binding_df, drop_columns=('Model', 'Pose'))
        row = {'model': model}
        row.update({f'binding_{column}_mean': value for column, value in means.items()})
        row['binding_n_structures'] = len(binding_df)
        rows.append(row)

    return pd.DataFrame(rows).sort_values('model').reset_index(drop=True)


def build_summary_dataframe() -> pd.DataFrame:
    af3_df = pd.read_csv(AF3_BEST_CSV)
    rosetta_df = _summarize_rosetta_scores()
    binding_df = _summarize_binding_energy()

    merged = af3_df.merge(rosetta_df, on='model', how='inner', validate='one_to_one')
    merged = merged.merge(binding_df, on='model', how='inner', validate='one_to_one')
    merged = merged.sort_values('model').reset_index(drop=True)

    expected_models = sorted(af3_df['model'].tolist())
    observed_models = sorted(merged['model'].tolist())
    if observed_models != expected_models:
        missing = sorted(set(expected_models) - set(observed_models))
        extra = sorted(set(observed_models) - set(expected_models))
        raise ValueError(
            'Merged summary does not match AF3 models. '
            f'Missing: {missing or "none"}. Extra: {extra or "none"}.'
        )

    return merged


## Construcció i validació del dataframe final

Es construeix el dataframe integrat cridant build_summary_dataframe, que realitza els tres passos de càrrega i els dos merges successius i verifica que el conjunt de models del CSV final coincideixi exactament amb el de les prediccions d'AF3.

In [4]:
summary_df = build_summary_dataframe()
summary_df.head()


,model,seed,sample,ranking_score,summary_chain_iptm,summary_chain_pair_iptm,summary_chain_pair_pae_min,summary_chain_ptm,summary_fraction_disordered,summary_has_clash,...,binding_interface_dG_A_B_mean,binding_interface_delta_sasa_A_B_mean,binding_interface_packstat_A_B_mean,binding_interface_delta_hbond_unsat_A_B_mean,binding_interface_nres_A_B_mean,binding_interface_hbonds_A_B_mean,binding_interface_sc_A_B_mean,binding_interface_dG_dSASA_ratio_A_B_mean,binding_interface_total_hbond_E_A_B_mean,binding_n_structures
0,Complex1,1,3,0.857156,"[0.82, 0.82]","[[0.75, 0.82], [0.82, 0.83]]","[[0.76, 1.31], [1.2, 0.76]]","[0.75, 0.83]",0.06,0.0,...,-49.306331,1532.050808,0.636095,3.000000,68.747748,4.405405,0.688180,-0.032202,-4.079136,111
1,Complex10,1,1,0.419977,"[0.34, 0.34]","[[0.74, 0.34], [0.34, 0.69]]","[[0.76, 7.28], [5.67, 0.76]]","[0.74, 0.69]",0.05,0.0,...,-53.922781,2305.289670,0.600464,10.423423,77.234234,4.171171,0.547139,-0.023371,-4.226829,111
2,Complex11,1,4,0.675917,"[0.62, 0.62]","[[0.75, 0.62], [0.62, 0.89]]","[[0.76, 3.36], [2.73, 0.76]]","[0.75, 0.89]",0.03,0.0,...,-69.924432,2340.876491,0.563191,7.756757,79.261261,8.270270,0.632037,-0.029896,-10.771509,111
3,Complex12,1,1,0.573140,"[0.52, 0.52]","[[0.71, 0.52], [0.52, 0.89]]","[[0.76, 5.8], [5.59, 0.76]]","[0.71, 0.89]",0.03,0.0,...,-57.302907,1927.894084,0.580929,12.279279,68.054054,5.405405,0.630147,-0.029734,-5.263209,111
4,Complex13,1,4,0.658324,"[0.61, 0.61]","[[0.74, 0.61], [0.61, 0.85]]","[[0.76, 3.15], [2.97, 0.76]]","[0.74, 0.85]",0.05,0.0,...,-55.906327,1719.882655,0.617424,9.621622,65.270270,12.153153,0.673121,-0.032524,-14.413007,111


## Exportació

El dataframe final es guarda com a final_bbf14.csv. Aquest fitxer és l'input de l'anàlisi estadística.

In [5]:
summary_df.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(summary_df)} complexes to {OUTPUT_CSV}')


Wrote 14 complexes to /Users/bertaguiu/projects/Scripts/final_bbf14.csv
